# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Content older than 12 months experiences a sharp decline in search visibility.

How exactly is this "sharp decline" labeled? Is it measured as a strict, continuous percentage drop, or is it an engineered bucket based on an arbitrary threshold? I would want to know if the label comes directly from raw measurement or subjective framing.

Finding 2: Refreshing content recovers lost traffic in 85% of cases.

Does the validation design isolate the refresh event from natural seasonality? I would ask if the observed recovery is a direct result of the update, or just directional momentum from a peak search season for those specific domains.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Here, I compare a standard random split (which risks data leakage by mixing a single client's articles across both train and test sets) against a grouped split (which strictly keeps clients isolated). The grouped split provides an honest measure of how the model will perform on an entirely unseen client.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score
from IPython.display import display

df = pd.read_csv("df_ranked.csv")
features = ['impressions_90d', 'content_age_days', 'days_since_last_update', 'avg_position', 'ctr']
X = df[features].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

# BEFORE
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_naive = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_acc = accuracy_score(y_test_naive, rf_naive.predict(X_test_naive))

# AFTER
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_group, X_test_group = X.iloc[train_idx], X.iloc[test_idx]
y_train_group, y_test_group = y.iloc[train_idx], y.iloc[test_idx]

rf_group = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_group.fit(X_train_group, y_train_group)
group_acc = accuracy_score(y_test_group, rf_group.predict(X_test_group))

# COMAPRING
comparison = pd.DataFrame({
    'Split Design': ['Random Split (Leakage Risk)', 'Grouped Split (Honest)'],
    'Accuracy': [naive_acc, group_acc]
})
display(comparison)

,Split Design,Accuracy
0,Random Split (Leakage Risk),0.658667
1,Grouped Split (Honest),0.568717


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

A leakage audit checks if we accidentally included features that give away the answer. Because our target label (is_declining_label) is inherently tied to traffic trends, we must ensure we aren't training the model on direct proxies for the answer (like trend_pct).

In [2]:
numeric_df = df.select_dtypes(include=['number'])
correlations = numeric_df.corrwith(df['is_declining_label']).sort_values(ascending=False)

print("Top Positive Correlations with Target:")
display(correlations.head(5))

print("\nTop Negative Correlations with Target:")
display(correlations.tail(5))

leak_threshold = 0.70
leaks = correlations[(abs(correlations) > leak_threshold) & (correlations.index != 'is_declining_label')]

if not leaks.empty:
    print(f"\nWARNING: Potential leakage detected in features:\n{leaks}")
else:
    print("\nNo obvious numeric leakage detected above 0.70 correlation threshold.")

Top Positive Correlations with Target:


is_declining_label        1.000000
days_with_impressions     0.190055
baseline_score            0.140381
word_count                0.090157
days_since_last_update    0.081383
dtype: float64


Top Negative Correlations with Target:


clicks_last_30d        -0.071935
impressions_last_30d   -0.093980
trend_pct              -0.141068
age_tier_order         -0.156142
content_age_days       -0.163882
dtype: float64


No obvious numeric leakage detected above 0.70 correlation threshold.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Unsafe Claim: "This model accurately predicts which articles will lose traffic, proving that outdated content causes ranking drops."

Rewritten Safe Claim: "This model provides directional decision-support, indicating that higher content age is frequently observed alongside measured ranking drops."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.